# Text Move Classifier — Multi-Language SAN Parser (TFLite)

**Purpose:**
Train a classifier that recognizes chess pieces from **move text notation** in multiple languages.

**Input:** Chess PDF with text-based notation (no figurine symbols)
- English: Kd4, Nxf6, e4
- French: Rd4, Cxf6, e4 (Roque, Cavalier)
- German: Kd4, Sxf6, e4 (Springer)
- Spanish: Rd4, Cxf6, e4 (Caballo)
- Italian: Rd4, Cxf6, e4 (Cavallo)

**Output:** `text_classifier.tflite` (~500 KB)

**Model:**
- Input: Move text (variable length, character-level)
- Output: Piece class (K/Q/R/B/N/P)
- Handles multiple languages via training data
- Size: < 500 KB (float32 TFLite)

## Step 1: Install Dependencies

In [ ]:
!pip install -q pdfplumber pillow numpy tensorflow matplotlib scikit-learn chess

## Step 2: Configuration

In [ ]:
# ── Classes ────────────────────────────────────────────────────────────────
PIECE_CLASSES = ['K', 'Q', 'R', 'B', 'N', 'P']   # piece letter

# ── Move text character encoding ───────────────────────────────────────────
# All possible characters in move notation (across all supported languages)
CHARSET = 'KQRBNPkqrbnpCDHSAcdhsaexX12345678+-#=!?'
CHAR_TO_IDX = {c: i for i, c in enumerate(CHARSET)}
IDX_TO_CHAR = {i: c for i, c in enumerate(CHARSET)}
VOCAB_SIZE = len(CHARSET)
MAX_MOVE_LEN = 8  # longest move notation (e.g., "Qxa7+!?")

# ── PDF Configuration ─────────────────────────────────────────────────────
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'  # EDIT: your PDF path
START_PAGE = 1
END_PAGE = 20       # extract from first 20 pages
MIN_SAMPLES_PER_PIECE = 50  # minimum training samples per piece

# ── Language mappings (piece letter → piece type across languages) ────────
LANGUAGE_MAPPINGS = {
    'english': {'K': 'K', 'Q': 'Q', 'R': 'R', 'B': 'B', 'N': 'N', 'P': 'P'},
    'french': {'R': 'R', 'D': 'Q', 'T': 'R', 'F': 'B', 'C': 'N', 'P': 'P'},
    'german': {'K': 'K', 'D': 'Q', 'T': 'R', 'L': 'B', 'S': 'N', 'B': 'P'},
    'spanish': {'R': 'R', 'D': 'Q', 'T': 'R', 'A': 'B', 'C': 'N', 'P': 'P'},
    'italian': {'R': 'R', 'D': 'Q', 'T': 'R', 'A': 'B', 'C': 'N', 'P': 'P'},
}

# ── Training ──────────────────────────────────────────────────────────────
EPOCHS = 50
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TFLITE_PATH = 'text_classifier.tflite'

print('Piece classes:', PIECE_CLASSES)
print('Charset:', CHARSET)
print('Max move length:', MAX_MOVE_LEN)
print('Vocab size:', VOCAB_SIZE)
print('PDF path:', PDF_PATH)

## Step 0: Mount Google Drive and Verify PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
    print(f'   Available items in /content/gdrive/MyDrive:')
    for item in os.listdir('/content/gdrive/MyDrive')[:20]:
        print(f'     - {item}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 3: Extract Move Text from PDF

In [ ]:
import pdfplumber
import re
from collections import defaultdict

# ── Filter: is this word a chess move? ────────────────────────────────────
def looks_like_chess_move(text):
    """Check if text looks like algebraic notation chess move."""
    clean = text.strip().upper()
    
    # Must contain file (a-h) and rank (1-8)
    has_file = bool(re.search(r'[A-H]', clean))
    has_rank = bool(re.search(r'[1-8]', clean))
    if not (has_file and has_rank):
        return False
    
    # Match move pattern: optional piece + file + optional x + file + rank + annotations
    move_pattern = r'^[KQRBNCDTFSHLAB]?[A-H]x?[A-H][1-8][+#=]?[!?]*$'
    if not re.match(move_pattern, clean):
        return False
    
    # Reject very long words
    if len(clean) > 12:
        return False
    
    return True

# ── Infer piece type from move text (multi-language) ──────────────────────
def infer_piece_from_text(text):
    """
    Infer which piece the move involves.
    Tries all language mappings to find a match.
    Returns piece class (K/Q/R/B/N/P) or None.
    """
    clean = text.strip().upper()
    first = clean[0] if clean else None
    
    if not first:
        return None
    
    # Try all language mappings
    for lang, mapping in LANGUAGE_MAPPINGS.items():
        if first in mapping:
            return mapping[first]
    
    # Fallback: if no piece letter detected, it's a pawn move
    if first in 'ABCDEFGH':
        return 'P'
    
    return None

# ── Extract moves from PDF ────────────────────────────────────────────────
moves_extracted = []  # {move_text, piece_class}

print(f'Extracting move text from pages {START_PAGE} to {END_PAGE}...\n')

with pdfplumber.open(PDF_PATH) as pdf:
    pdf_page_count = len(pdf.pages)
    if END_PAGE > pdf_page_count:
        END_PAGE = pdf_page_count
        print(f'  (PDF has {pdf_page_count} pages, adjusted END_PAGE)\n')
    
    for page_idx in range(START_PAGE - 1, min(END_PAGE, pdf_page_count)):
        pdf_page = pdf.pages[page_idx]
        print(f'Processing page {page_idx + 1}...')
        
        # Extract words
        try:
            words = pdf_page.extract_words()
        except:
            print(f'  ⚠ Could not extract words')
            continue
        
        if not words:
            print(f'  ⚠ No words found')
            continue
        
        page_count = 0
        for word in words:
            text = word.get('text', '')
            
            # Filter: only process chess moves
            if not looks_like_chess_move(text):
                continue
            
            # Infer piece type
            piece = infer_piece_from_text(text)
            if not piece:
                continue
            
            moves_extracted.append({
                'text': text,
                'piece': piece,
                'page': page_idx + 1,
            })
            page_count += 1
        
        print(f'  ✅ Extracted {page_count} moves\n')

print(f'📊 Summary:')
print(f'  Total moves extracted: {len(moves_extracted)}')

piece_counts = defaultdict(int)
for move in moves_extracted:
    piece_counts[move['piece']] += 1

print(f'\nMoves by piece type:')
for piece in PIECE_CLASSES:
    count = piece_counts[piece]
    status = '✅' if count >= MIN_SAMPLES_PER_PIECE else '⚠'
    if count > 0:
        print(f'  {status} {piece}: {count}')

if len(moves_extracted) == 0:
    print('\n❌ No moves extracted!')
    print('   Check: PDF path, page range, move notation')

## Step 4: Build Training Dataset

In [ ]:
import numpy as np

# ── Encode move text to indices ────────────────────────────────────────────
def encode_move(move_text, max_len=MAX_MOVE_LEN):
    """
    Encode move text as fixed-length vector of character indices.
    Pads with 0 (special index) on the right.
    """
    encoded = []
    for char in move_text.upper()[:max_len]:
        if char in CHAR_TO_IDX:
            encoded.append(CHAR_TO_IDX[char])
    # Pad with 0
    while len(encoded) < max_len:
        encoded.append(0)
    return np.array(encoded[:max_len], dtype=np.int32)

# ── Build dataset ────────────────────────────────────────────────────────
X, y = [], []
samples_per_piece = defaultdict(int)

for move in moves_extracted:
    text = move['text']
    piece = move['piece']
    piece_idx = PIECE_CLASSES.index(piece)
    
    # Encode move text
    encoded = encode_move(text)
    X.append(encoded)
    y.append(piece_idx)
    samples_per_piece[piece] += 1

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)

print(f'Dataset shape: X={X.shape}  y={y.shape}')
print(f'\nTraining samples per piece:')
for piece in PIECE_CLASSES:
    count = samples_per_piece[piece]
    status = '✅' if count >= MIN_SAMPLES_PER_PIECE else '⚠'
    if count > 0:
        print(f'  {status} {piece}: {count}')

if len(X) < 100:
    print(f'\n⚠ Warning: Only {len(X)} samples. Consider extending END_PAGE.')

## Step 5: Split Data and Build Model

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(MAX_MOVE_LEN,)),
    
    # Character embedding
    tf.keras.layers.Embedding(VOCAB_SIZE + 1, 16, input_length=MAX_MOVE_LEN),
    
    # 1D convolution for character-level features
    tf.keras.layers.Conv1D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    
    # Dense layers
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(PIECE_CLASSES), activation='softmax'),
], name='text_move_classifier')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## Step 6: Train Model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, mode='max', monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5, monitor='val_accuracy'),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest Accuracy: {test_acc:.2%}')

## Step 7: Training Curves and Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy')
ax1.legend()
ax1.set_xlabel('epoch')
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss')
ax2.legend()
ax2.set_xlabel('epoch')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Confusion matrix
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=PIECE_CLASSES).plot(ax=ax, colorbar=False)
plt.title('Test Confusion Matrix')
plt.tight_layout()
plt.show()

print('\nPer-class accuracy:')
for i, piece in enumerate(PIECE_CLASSES):
    if cm[i].sum() > 0:
        acc = cm[i, i] / cm[i].sum()
        print(f'  {piece}: {acc:.1%}')

## Step 8: Convert to TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter.inference_input_type = tf.int32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'✅ Model saved: {TFLITE_PATH}')
print(f'   Size: {size_kb:.0f} KB')
print(f'   Accuracy: {test_acc:.1%}')
print(f'   Classes: {len(PIECE_CLASSES)}')

## Step 9: Download Model

In [ ]:
from google.colab import files

if os.path.exists(TFLITE_PATH):
    files.download(TFLITE_PATH)
    print(f'✅ Downloaded {TFLITE_PATH}')
else:
    print(f'❌ Model file not found: {TFLITE_PATH}')